# LLM From Scratch — `the-verdict.txt` training + visual debugger

This notebook is the Kaggle workflow for our `llm-visualize` project.

It uses Sebastian Raschka's **official Chapter 5 code** for the GPT model and training loop, then uses our tracer/visualizer after training.

**Chapters 6 and 7 are not needed** for this project; they are fine-tuning chapters.

Workflow:

`the-verdict.txt -> train GPT -> save checkpoint -> trace one forward pass -> render debugger`

> Kaggle: enable **Internet** for the first run so the notebook can clone the two GitHub repositories.

## 1. Install dependency and clone the repositories

In [ ]:
!pip -q install tiktoken

import os
import sys
import subprocess

WORKING = "/kaggle/working"
RASCHKA_DIR = f"{WORKING}/LLMs-from-scratch"
VIS_DIR = f"{WORKING}/llm-visualize"

def clone_once(url, target):
    if not os.path.exists(target):
        subprocess.run(["git", "clone", "--depth", "1", url, target], check=True)
    else:
        print(f"Already exists: {target}")

clone_once("https://github.com/rasbt/LLMs-from-scratch.git", RASCHKA_DIR)
clone_once("https://github.com/gerasidev/llm-visualize.git", VIS_DIR)

CH05_DIR = f"{RASCHKA_DIR}/ch05/01_main-chapter-code"
sys.path.insert(0, CH05_DIR)
sys.path.insert(0, VIS_DIR)

print("Raschka code:", CH05_DIR)
print("Visualizer:", VIS_DIR)

## 2. Import Raschka's model/training code and our tracing code

In [ ]:
import torch
import tiktoken

from previous_chapters import GPTModel, create_dataloader_v1
from gpt_train import train_model_simple, generate_and_print_sample

from python.trace_raschka import (
    save_model_checkpoint,
    load_model_checkpoint,
    trace_forward_pass,
    write_trace,
)
from python.export_visualizer import build_visualizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Book configuration

This is the Chapter 5 GPT-124M configuration Raschka uses, with context length shortened to 256 for this training example.

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

TRAIN_SETTINGS = {
    "learning_rate": 5e-4,
    "num_epochs": 10,
    "batch_size": 2,
    "weight_decay": 0.1,
}

# First run: leave this as None and train.
# Future Kaggle session: point this to an attached saved checkpoint, for example:
# CHECKPOINT_TO_LOAD = "/kaggle/input/my-verdict-checkpoint/verdict-gpt.pth"
CHECKPOINT_TO_LOAD = None

CHECKPOINT_OUT = "/kaggle/working/verdict-gpt.pth"
TRACE_OUT = "/kaggle/working/trace.json"
HTML_OUT = "/kaggle/working/llm-debugger.html"

## 4. Load `the-verdict.txt` and create train/validation data

In [ ]:
verdict_path = f"{RASCHKA_DIR}/ch02/01_main-chapter-code/the-verdict.txt"

with open(verdict_path, "r", encoding="utf-8") as f:
    text_data = f.read()

train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))

train_loader = create_dataloader_v1(
    text_data[:split_idx],
    batch_size=TRAIN_SETTINGS["batch_size"],
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0,
)

val_loader = create_dataloader_v1(
    text_data[split_idx:],
    batch_size=TRAIN_SETTINGS["batch_size"],
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0,
)

tokenizer = tiktoken.get_encoding("gpt2")

print("Characters:", len(text_data))
print("Tokens:", len(tokenizer.encode(text_data)))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

## 5. Create model / optionally load an existing checkpoint

If `CHECKPOINT_TO_LOAD` points to a saved model, training is skipped.

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M).to(device)

checkpoint_loaded = False

if CHECKPOINT_TO_LOAD:
    if not os.path.exists(CHECKPOINT_TO_LOAD):
        raise FileNotFoundError(CHECKPOINT_TO_LOAD)
    metadata = load_model_checkpoint(model, CHECKPOINT_TO_LOAD, device=device)
    checkpoint_loaded = True
    print("Loaded checkpoint:", CHECKPOINT_TO_LOAD)
    print("Metadata:", metadata)
else:
    print("No checkpoint selected. The next cell will train from scratch.")

## 6. Train from scratch — run only when you do not already have a checkpoint

This is the slow cell. Once it finishes, the model is saved to `/kaggle/working/verdict-gpt.pth`.

After that, UI/debugger changes do **not** require training again.

In [ ]:
if not checkpoint_loaded:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAIN_SETTINGS["learning_rate"],
        weight_decay=TRAIN_SETTINGS["weight_decay"],
    )

    train_losses, val_losses, tokens_seen = train_model_simple(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        device=device,
        num_epochs=TRAIN_SETTINGS["num_epochs"],
        eval_freq=5,
        eval_iter=1,
        start_context="Every effort moves you",
        tokenizer=tokenizer,
    )

    save_model_checkpoint(
        model,
        CHECKPOINT_OUT,
        config=GPT_CONFIG_124M,
        optimizer=optimizer,
        extra={"dataset": "the-verdict.txt", "epochs": TRAIN_SETTINGS["num_epochs"]},
    )
    print("Saved:", CHECKPOINT_OUT)
else:
    print("Checkpoint already loaded — skipping training.")

## 7. Quick sanity check: generate text with the trained model

In [ ]:
generate_and_print_sample(
    model=model,
    tokenizer=tokenizer,
    device=device,
    start_context="Every effort moves you",
)

## 8. Trace one short forward pass

This is the important part for the visualization. It does **not** train the model.

In [ ]:
TRACE_TEXT = "Every effort moves you"

ids = tokenizer.encode(TRACE_TEXT)
input_ids = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)

trace = trace_forward_pass(
    model,
    input_ids,
    tokens=[tokenizer.decode([i]) for i in ids],
    decode_token=lambda i: tokenizer.decode([i]),
    top_k=10,
)

write_trace(trace, TRACE_OUT)

print("Trace saved:", TRACE_OUT)
print("Tokens:", [(t["text"], t["id"]) for t in trace["tokens"]])
print("Layers traced:", len(trace["layers"]))

## 9. Build the standalone HTML debugger

In [ ]:
build_visualizer(
    trace_path=TRACE_OUT,
    output_path=HTML_OUT,
)

print("Debugger saved:", HTML_OUT)

## 10. Display the debugger in Kaggle

If Kaggle blocks JavaScript in the inline rendering, download/open `llm-debugger.html` from notebook outputs instead.

In [ ]:
from IPython.display import HTML, display, FileLink

with open(HTML_OUT, "r", encoding="utf-8") as f:
    display(HTML(f.read()))

display(FileLink(CHECKPOINT_OUT if os.path.exists(CHECKPOINT_OUT) else CHECKPOINT_TO_LOAD))
display(FileLink(TRACE_OUT))
display(FileLink(HTML_OUT))